Files originally from https://github.com/LucasSilvaFerreira/Perturb_Loader

In [1]:
import mudata as md
import anndata as ad
import numpy as np
import pandas as pd
import perturbvi
import os
import pyro

smoke_test = True


/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Global seed set to 0


In [ ]:
def construct_mudata(data_dir = "."):
    rna_adata = ad.read_h5ad(f"{data_dir}/ann_exp.h5ad")
    rna_adata.obs['library_size']=rna_adata.X.sum(axis=1)
    rna_adata.varm['element_tested'] = ad.read_h5ad(f"{data_dir}/ann_Element_x_tested_genes.h5ad").to_df().T
    grna_adata = ad.read_h5ad(f"{data_dir}/ann_guide.h5ad")
    grna_adata.varm['element_targeted'] = ad.read_h5ad(f"{data_dir}/ann_Element_guide.h5ad").to_df().T
    assert (rna_adata.varm['element_tested'].columns == grna_adata.varm['element_targeted'].columns).all()
    mdata = md.MuData({'rna':rna_adata, 'grna':grna_adata})
    mdata.write_h5mu(f'{data_dir}/gasperini_pilot_highMOI.h5mu')
    return mdata

force = True
mudata_file = "gasperini_pilot_highMOI.h5mu"
data_dir = "../../../../Data/gasperini_pilot"
if mudata_file not in os.listdir(data_dir) or force:
    mdata = construct_mudata(data_dir)
else:
    mdata = md.read_h5mu(os.path.join(data_dir, mudata_file))

In [ ]:
# mdata['grna'].varm['gene_targeted']
# mdata['rna'].varm['gene_tested']
mdata['grna'].varm['element_targeted']

Element,ACTB_TSS,ACTG1_TSS,ACYP1_TSS,ADIPOR1_TSS,ALDH1A2_TSS,APEX1_TSS,APLP2_TSS,ARF6_TSS,ARID1A_TSS,ARL4A_TSS,...,scrambled_24,scrambled_25,scrambled_2,scrambled_3,scrambled_4,scrambled_5,scrambled_6,scrambled_7,scrambled_8,scrambled_9
ACTB_TSS|1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACTB_TSS|2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACTG1_TSS|1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACTG1_TSS|2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACYP1_TSS|1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
scrambled_7|2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
scrambled_8|1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
scrambled_8|2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
scrambled_9|1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [ ]:
guides = mdata['grna'].var_names
selected_guides = list(guides[guides.str.contains(r"TSS|random|scrambled", regex=True)])
if smoke_test:
    selected_guides = selected_guides[:20] + selected_guides[-10:]
    print("guides", selected_guides)

selected_elements = list({guide.split('|')[0] for guide in selected_guides if '_TSS' in guide})
element_subset = mdata['grna'].varm['element_targeted'][selected_elements].copy()

grna_subset = mdata['grna']
grna_subset.varm['element_targeted'] = element_subset.values

grna_subset = mdata['grna'][:,selected_guides]
genes = list({e.split('_')[0] for e in selected_elements if '_TSS' in e})
subset_genes = [g for g in genes if g in mdata['rna'].var_names]

if smoke_test:
    print("elements",selected_elements)
    print("genes",subset_genes)
assert grna_subset.varm['element_targeted'].shape == (len(selected_guides), len(selected_elements))


guides ['ACTB_TSS|1', 'ACTB_TSS|2', 'ACTG1_TSS|1', 'ACTG1_TSS|2', 'ACYP1_TSS|1', 'ACYP1_TSS|2', 'ADIPOR1_TSS|1', 'ADIPOR1_TSS|2', 'ALDH1A2_TSS|1', 'ALDH1A2_TSS|2', 'APEX1_TSS|1', 'APEX1_TSS|2', 'APLP2_TSS|1', 'APLP2_TSS|2', 'ARF6_TSS|1', 'ARF6_TSS|2', 'ARID1A_TSS|1', 'ARID1A_TSS|2', 'ARL4A_TSS|1', 'ARL4A_TSS|2', 'scrambled_5|1', 'scrambled_5|2', 'scrambled_6|1', 'scrambled_6|2', 'scrambled_7|1', 'scrambled_7|2', 'scrambled_8|1', 'scrambled_8|2', 'scrambled_9|1', 'scrambled_9|2']
elements ['ACTB_TSS', 'ARL4A_TSS', 'ALDH1A2_TSS', 'ARF6_TSS', 'APEX1_TSS', 'ARID1A_TSS', 'APLP2_TSS', 'ACTG1_TSS', 'ADIPOR1_TSS', 'ACYP1_TSS']
genes ['ARF6', 'ARL4A', 'ARID1A', 'APEX1', 'ACTG1', 'ADIPOR1', 'ACYP1', 'APLP2', 'ALDH1A2', 'ACTB']


In [ ]:
grna_subset.varm['element_targeted'].shape

(30, 10)

In [ ]:
# subset down to only control perturbed genes
# rna_subset = mdata['rna'][:,mdata['rna'].varm['gene_tested'].sum(axis=1).values > 0]
rna_subset =  mdata['rna'][:,subset_genes]
mdata_subset = md.MuData({'rna':rna_subset.copy(), 'grna': grna_subset.copy()})
# assert (mdata_subset['rna'].varm['element_tested'].columns == mdata_subset['grna'].varm['element_targeted'].columns).all()
mdata_subset

MuData object with n_obs × n_vars = 47964 × 40
  2 modalities
    rna:	47964 x 10
      obs:	'bath_number', 'percent_mito', 'log_number_of_detected_genes', 'log_total_gene_count', 'log_total_guide_count', 'library_size'
      varm:	'element_tested'
    grna:	47964 x 30
      obs:	'bath_number', 'percent_mito', 'log_number_of_detected_genes', 'log_total_gene_count', 'log_total_guide_count'
      varm:	'element_targeted'

In [ ]:
perturbvi.PERTURBVI.setup_mudata(
    mdata_subset,
    batch_key="bath_number",
    size_factor_key="library_size",
    element_key="element_targeted",
    modalities={
        "rna_layer": 'rna',
        "perturbation_layer": 'grna',
    },
)

model = perturbvi.PERTURBVI(mdata_subset)
model.view_anndata_setup()

INFO     Generating sequential column names                                                                        


Anndata setup with scvi-tools version 0.20.1.

Setup via `PERTURBVI.setup_anndata` with arguments:

{
│   'rna_layer': None,
│   'batch_key': 'bath_number',
│   'element_key': 'element_targeted',
│   'perturbation_layer': None,
│   'modalities': {'rna_layer': 'rna', 'perturbation_layer': 'grna'},
│   'size_factor_key': 'library_size'
}

     Summary Statistics     
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Summary Stat Key ┃ Value ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│     n_batch      │   6   │
│     n_cells      │ 47964 │
│    n_elements    │  10   │
│ n_perturbations  │  30   │
│      n_vars      │  10   │
└──────────────────┴───────┘

                          Data Registry                           
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃   Registry Key    ┃            scvi-tools Location             ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         X         │             adata.mod['rna'].X             │
│       batch       │    adata.mod['rna'].obs['_scvi_batch']     │
│     elements      │ adata.mod['grna'].varm['element_targeted'] │
│       ind_x       │       adata.mod['rna'].obs['_ind_x']       │
│ observed_lib_size │    adata.mod['rna'].obs['library_size']    │
│   perturbations   │            adata.mod['grna'].X             │
└───────────────────┴────────────────────────────────────────────┘

                     batch State Registry                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃     Source Location      ┃ Categories ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['bath_number'] │     1      │          0          │
│                          │     2      │          1          │
│                          │     3      │          2          │
│                          │     4      │          3          │
│                          │     5      │          4          │
│                          │     6      │          5          │
└──────────────────────────┴────────────┴─────────────────────┘

In [ ]:
# optimizer = pyro.optim.ClippedAdam({'lr':0.001, 'lrd':0.9})
model.train(
    max_epochs=100,
    lr=0.1,
    batch_size=4096,
    # plan_kwargs={"optim": optimizer},
)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning: MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.
  rank_zero_warn(
/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pytorch_lightning/trainer/configuration_validator.py:106: UserWarning: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
  rank_zero_warn("You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.")


Epoch 100/100: 100%|██████████| 100/100 [00:15<00:00,  6.83it/s, v_num=1, elbo_train=9.79e+5]

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 100/100: 100%|██████████| 100/100 [00:15<00:00,  6.65it/s, v_num=1, elbo_train=9.79e+5]


In [ ]:
%load_ext autoreload
%autoreload 2
from scipy.stats import norm

# lfc_threshold = 0.1 # ~ 10% knockdown
lfc_threshold = 0

if smoke_test:
    for gene_index in range(len(subset_genes)):
        print(mdata_subset['rna'].var_names[gene_index])

        perturb_mean_lfc_mu = pyro.get_param_store()['perturb_mean_lfc.mu'].detach().cpu().numpy()
        perturb_disp_lfc_mu = pyro.get_param_store()['perturb_disp_lfc.mu'].detach().cpu().numpy()
        lfc_cov_tril = pyro.get_param_store()['perturb_lfc.scale_tril'] * pyro.get_param_store()['scale_factor'].exp()
        lfc_cov = lfc_cov_tril @ lfc_cov_tril.transpose(dim0=-1, dim1=-2)
        perturb_mean_lfc_sigma = lfc_cov[...,0,0].sqrt().detach().cpu().numpy()
        perturb_disp_lfc_sigma = lfc_cov[...,0,0].sqrt().detach().cpu().numpy()
        assert perturb_mean_lfc_mu.shape == perturb_mean_lfc_sigma.shape
        perturb_z_scores = perturb_mean_lfc_mu/perturb_mean_lfc_sigma
        perturb_p_vals = norm.cdf(lfc_threshold, loc=-perturb_mean_lfc_mu, scale=perturb_mean_lfc_sigma)

        # perturb_z_scores[:,gene_index].detach().cpu().numpy()
        z_df = pd.DataFrame({'z_score': perturb_z_scores[:,gene_index],
                            'mean_mu': perturb_mean_lfc_mu[:,gene_index],
                            'disp_mu': perturb_disp_lfc_mu[:,gene_index],
                            'mu_p_val': perturb_p_vals[:,gene_index],
                            'grna': mdata_subset['grna'].var_names,})
        # z_df.hist('mu_p_val', bins=100)
        # z_df.plot(x='mean_mu',y='disp_mu', style='o')
        print(z_df.sort_values('mu_p_val', ascending=True).reset_index(drop=True).head(20))


ARF6
      z_score   mean_mu   disp_mu      mu_p_val           grna
0  -17.163586 -0.602229  0.124381  2.487268e-66     ARF6_TSS|2
1  -12.030066 -0.422106  0.032918  1.234809e-33     ARF6_TSS|1
2   -7.588734 -0.266270 -0.098105  1.615235e-14     ACTB_TSS|1
3   -7.059071 -0.247686 -0.104911  8.380967e-13  scrambled_7|2
4   -6.584565 -0.231037 -0.231874  2.281100e-11  scrambled_5|2
5   -6.060224 -0.212639 -0.082847  6.796603e-10    ACYP1_TSS|2
6   -5.718186 -0.200637 -0.149141  5.383351e-09    ACTG1_TSS|1
7   -4.742863 -0.166416 -0.136043  1.053593e-06  scrambled_5|1
8   -3.919271 -0.137518 -0.225927  4.440856e-05  ALDH1A2_TSS|2
9   -3.203107 -0.112389 -0.032917  6.797676e-04    ACYP1_TSS|1
10  -3.095282 -0.108606  0.004170  9.831290e-04     ACTB_TSS|2
11  -2.315232 -0.081236  0.101327  1.030012e-02    ARL4A_TSS|2
12  -1.853026 -0.065018  0.056643  3.193935e-02  scrambled_8|1
13  -1.730432 -0.060717  0.013941  4.177656e-02  ADIPOR1_TSS|2
14  -1.627573 -0.057108 -0.061071  5.180777e-02  s

In [ ]:
for k, v in pyro.get_param_store().items():
    print (k, v.shape)

scale_factor torch.Size([])
log_var_mean.mu torch.Size([10])
log_var_disp.mu torch.Size([10])
batch_effect.mu torch.Size([6, 1])
batch_effect.sigma torch.Size([6, 1])
log_var_mean.sigma torch.Size([10])
log_var_disp.sigma torch.Size([10])
perturb_mean_lfc.mu torch.Size([30, 10])
perturb_disp_lfc.mu torch.Size([30, 10])
perturb_lfc.scale_tril torch.Size([30, 10, 2, 2])
